<a href="https://colab.research.google.com/github/krishnareddyalavala/Data-Science-models/blob/main/Evaluation_of_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask prometheus-client scikit-learn numpy requests

In [ ]:
# -------------------------
import time, random, threading
import numpy as np
from flask import Flask, request, jsonify
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from prometheus_client import (
    Counter, Histogram, Gauge, start_http_server
)

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)

In [ ]:
REQUEST_COUNT = Counter(
    "ml_requests_total",
    "Total inference requests",
    ["endpoint"]
)

In [ ]:
REQUEST_LATENCY = Histogram(
    "ml_inference_latency_seconds",
    "Inference latency"
)

In [ ]:
MODEL_ACCURACY = Gauge(
    "ml_model_accuracy",
    "Current model accuracy"
)

In [ ]:
DATA_DRIFT = Gauge(
    "ml_data_drift_score",
    "Simple data drift score"
)

In [ ]:
MODEL_ACCURACY.set(accuracy)

In [ ]:
app = Flask(__name__)

@app.route("/predict", methods=["POST"])
def predict():
    REQUEST_COUNT.labels(endpoint="/predict").inc()

    with REQUEST_LATENCY.time():
        data = np.array(request.json["features"]).reshape(1, -1)
        prediction = model.predict(data)[0]
        time.sleep(random.uniform(0.05, 0.4))  # simulate inference delay

    # Fake drift metric
    drift_score = np.abs(np.mean(data) - np.mean(X_train))
    DATA_DRIFT.set(drift_score)

    return jsonify({
        "prediction": int(prediction),
        "model_accuracy": accuracy,
        "drift_score": float(drift_score)
    })

In [ ]:
def run_app():
    app.run(port=5000)

start_http_server(8000)  # Prometheus metrics
threading.Thread(target=run_app).start()

In [ ]:
import requests

sample = X_test[0].tolist()
for _ in range(40):
    try:
        requests.post(
            "http://localhost:5000/predict",
            json={"features": sample}
        )
    except:
        pass

print(" MLOps pipeline running")
print(" Inference API  : http://localhost:5000/predict")
print(" Metrics endpoint: http://localhost:8000/metrics")

INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:00] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:00] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:01] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:01] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:01] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:02] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:02] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:02] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:02] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:02] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:02] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [07/Apr/2026 04:05:02] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:12

 MLOps pipeline running
 Inference API  : http://localhost:5000/predict
 Metrics endpoint: http://localhost:8000/metrics
